LIBRARIES

In [1]:
# Set seed for reproducibility
SEED = 42

# Import necessary libraries
import os

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch

torch.manual_seed(SEED)
from torch import nn

# from torchsummary import summary


if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
import copy
import shutil
from itertools import product
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
#%matplotlib inline


PyTorch version: 2.9.0+cpu
Device: cpu


Data Analysis

In [3]:
# Define column names for the dataset
column_names = ['sample_index', 'time', 'pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4', 'n_legs',
                'n_hands', 'n_eyes', 'joint_00', 'joint_01', 'joint_02', 'joint_03', 'joint_04', 'joint_05', 'joint_06',
                'joint_07', 'joint_08', 'joint_09', 'joint_10', 'joint_11', 'joint_12', 'joint_13', 'joint_14',
                'joint_15', 'joint_16', 'joint_17', 'joint_18', 'joint_19', 'joint_20', 'joint_21', 'joint_22',
                'joint_23', 'joint_24', 'joint_25', 'joint_26', 'joint_27', 'joint_28', 'joint_29', 'joint_30']

# Read the dataset into a DataFrame with specified column names
df = pd.read_csv('pirate_pain_train.csv', header=None, names=column_names)

# Remove rows with any missing values
df.dropna(axis=0, how='any', inplace=True)

# Print the shape of the DataFrame
print(f"DataFrame shape: {df.shape}")

# Display the first 10 rows of the DataFrame
df.head(10)


DataFrame shape: (105761, 40)


,sample_index,time,pain_survey_1,pain_survey_2,pain_survey_3,pain_survey_4,n_legs,n_hands,n_eyes,joint_00,...,joint_21,joint_22,joint_23,joint_24,joint_25,joint_26,joint_27,joint_28,joint_29,joint_30
0,sample_index,time,pain_survey_1,pain_survey_2,pain_survey_3,pain_survey_4,n_legs,n_hands,n_eyes,joint_00,...,joint_21,joint_22,joint_23,joint_24,joint_25,joint_26,joint_27,joint_28,joint_29,joint_30
1,000,0,2,0,2,1,two,two,two,1.0947052308906111,...,3.499557792481375e-06,1.945042470132259e-06,3.999558416844207e-06,1.153299174805662e-05,3.8059302778858464e-06,0.017592085928338518,0.013507849980388193,0.026797650940792724,0.027814594224909676,0.5
2,000,1,2,2,2,2,two,two,two,1.135183103536129,...,3.976952213431784e-07,6.7651074426812e-07,6.019626875860215e-06,4.64377443665608e-08,0.0,0.013352218672651914,0.0,0.013376576038862383,0.013715933906977116,0.5
3,000,2,2,0,2,2,two,two,two,1.0807448109145028,...,1.5338202764856223e-07,1.6985249698772954e-07,1.4460506893440146e-06,2.424536490939014e-06,2.5135187612327057e-06,0.01622542717601379,0.008110441391572916,0.02409691522589848,0.023105023010497494,0.5
4,000,3,2,2,2,2,two,two,two,0.9380165318378043,...,1.0068651685544181e-05,5.511079228636987e-07,1.8475966646879033e-06,5.432416362110678e-08,0.0,0.011831559452139423,0.00745007119889528,0.028613141314700982,0.02464820836318195,0.5
5,000,4,2,2,2,2,two,two,two,1.0901849482346806,...,4.437265668725749e-06,1.7354585896225497e-07,1.5527219085486641e-06,5.8253658712538915e-08,7.044831533755251e-06,0.005360386839374858,0.002531546539790179,0.03302617453410233,0.02532797214906949,0.5
6,000,5,2,0,2,1,two,two,two,1.1460314612288625,...,1.0731674090588247e-06,1.7538372278427127e-07,2.957340042075233e-07,6.217310659686344e-08,7.475357955435864e-06,0.006150383917344993,0.006444462796853394,0.033101101838941854,0.02376662479054096,0.5
7,000,6,2,1,2,1,two,two,two,1.0258698643268944,...,1.074799677094579e-06,1.772156251347998e-07,1.9765583948385075e-06,1.5760863171953413e-06,4.637804261006082e-06,0.006495420711872876,0.006420783622156604,0.0318036627067577,0.01905554991156486,0.5
8,000,7,2,2,2,2,two,two,two,1.038597318187619,...,8.82907419971825e-07,1.7904150374561284e-07,2.2105618044813743e-06,1.4857409468201832e-06,0.0,0.015997945858351518,0.0053974619479166975,0.035551569733665286,0.015731614426066978,0.5
9,000,8,2,2,0,1,two,two,two,0.9842507543998211,...,1.6210552284637883e-06,1.1651607968521788e-06,3.0301636894037034e-07,5.416678423467074e-07,0.0,0.020538582101834763,0.008516687431520533,0.008635017533525661,0.015257299098127262,0.5


In [4]:
# Display a concise summary of the DataFrame
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105761 entries, 0 to 105760
Data columns (total 40 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   sample_index   105761 non-null  object
 1   time           105761 non-null  object
 2   pain_survey_1  105761 non-null  object
 3   pain_survey_2  105761 non-null  object
 4   pain_survey_3  105761 non-null  object
 5   pain_survey_4  105761 non-null  object
 6   n_legs         105761 non-null  object
 7   n_hands        105761 non-null  object
 8   n_eyes         105761 non-null  object
 9   joint_00       105761 non-null  object
 10  joint_01       105761 non-null  object
 11  joint_02       105761 non-null  object
 12  joint_03       105761 non-null  object
 13  joint_04       105761 non-null  object
 14  joint_05       105761 non-null  object
 15  joint_06       105761 non-null  object
 16  joint_07       105761 non-null  object
 17  joint_08       105761 non-null  object
 18  join

In [5]:
# Generate descriptive statistics for numerical columns in the DataFrame
df.describe()

,sample_index,time,pain_survey_1,pain_survey_2,pain_survey_3,pain_survey_4,n_legs,n_hands,n_eyes,joint_00,...,joint_21,joint_22,joint_23,joint_24,joint_25,joint_26,joint_27,joint_28,joint_29,joint_30
count,105761,105761,105761,105761,105761,105761,105761,105761,105761,105761.0,...,105761.0,1.057610e+05,105761.0,1.057610e+05,105761.0,105761.000000,105761.0,105761.0,105761.0,105761.0
unique,663,321,7,7,7,7,3,3,3,105721.0,...,68022.0,6.790500e+04,67515.0,6.715100e+04,69749.0,103847.000000,101836.0,104126.0,105278.0,3.0
top,660,159,2,2,2,2,two,two,two,0.0,...,0.0,3.375823e-07,0.0,2.204679e-07,0.0,0.000392,0.0,0.0,0.0,0.5
freq,160,559,67259,68453,68311,68868,104800,104800,104800,39.0,...,12807.0,2.320000e+02,1085.0,2.370000e+02,10407.0,25.000000,3380.0,1340.0,205.0,89377.0


Data Preprocessing